# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # single object, not dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using their Croissant `@id` fields.

In [ ]:
# List all RecordSet `@id`s and their fields from the dataset.
# Using dataset.record_sets (a list of RecordSet objects)

record_sets = dataset.record_sets
print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {getattr(rs, 'name', '<no name>')}")
    print(f"  RecordSet @id: {rs.id}")
    print(f"  Fields:")
    for field in getattr(rs, 'fields', []):
        col_ids = getattr(field, 'columns', [])
        if col_ids:
            col_ids = ', '.join([col.id for col in col_ids])
        else:
            col_ids = '<none>'
        print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', '<no name>')} | Data Type: {getattr(field, 'data_type', '<unknown>')} | Columns: {col_ids}")
    print()

## 3. Data Extraction
Load data from each RecordSet using their `@id` and field `@id` values. Data is loaded into pandas DataFrames for analysis.

In [ ]:
# Populate a list of RecordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Show info for the main record set (assume the first one is main clinically relevant table)
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"\nColumns for main RecordSet ({main_rs}):")
    if main_rs in dataframes:
        print(dataframes[main_rs].columns.tolist())
        dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
import numpy as np

# Use the main RecordSet and get a numeric field (by field @id, e.g. 'age')
main_rs = record_set_ids[0]
df = dataframes[main_rs].copy()

# Print all columns for exploration (mapped to their Croissant IDs)
print(f"Columns in main RecordSet ({main_rs}):\n{df.columns.tolist()}\n")

# Pick a numeric field. Let's try to find 'age' in various forms.
possible_numeric_fields = [c for c in df.columns if 'age' in c.lower() or df[c].dtype in [np.int64, np.float64]]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use first matching
    print(f"Using numeric field: {numeric_field_id}\n")
    # Use a generic threshold for demonstration
    threshold = df[numeric_field_id].mean()  # for demo, filter above mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
else:
    print("No numeric fields found for filtering.")

# Try grouping by a field: pick any categorical with few unique values (e.g. 'sex', 'gender', 'location', or similar)
possible_group_fields = [c for c in df.columns if df[c].dtype == object and df[c].nunique() < len(df) // 4]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"\nGrouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(grouped_df.head())
else:
    print("No suitable categorical grouping field found.")

## 5. Visualization
Visualize data distributions and relationships using pandas and matplotlib. Here we plot the numeric field distribution and the groupwise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if possible_numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped, plot groupwise means as barplot
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(6, 4))
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load dataset metadata and tabular data using the `mlcroissant` library by referencing all entities via their Croissant `@id`. We provided a brief data overview, filtered and normalized a numeric field, grouped by a categorical field, and produced basic visualizations. This workflow can be adapted for in-depth analysis and modeling using any Croissant-compliant dataset.
